### Imports

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Create Flag Parameter

In [0]:
dbutils.widgets.text('incremental_flag', '0')

In [0]:
incremental_flag = dbutils.widgets.get('incremental_flag')
print(incremental_flag)

### Creating Dimensions Model

#### Fetch Related Columns

In [0]:
df_src = spark.sql('''
                   SELECT DISTINCT (Branch_ID), BranchName
                   FROM parquet.`abfss://silver@cacdyamadls.dfs.core.windows.net/carsales`
                   ''')

df_src.display()

#### Dim Model Sink - Initial and Incremental Load

In [0]:
#If table does not exist create the schema
if spark.catalog.tableExists('cars_catalog.gold.dim_branch') :
    df_sink = spark.sql('''
    SELECT dim_branch_key, Brach_ID, BranchName
    FROM parquet.`abfss://silver@cacdyamadls.dfs.core.windows.net/carsales`
    ''')
else :
    df_sink = spark.sql('''
    SELECT 1 as dim_branch_key, Branch_ID, BranchName
    FROM parquet.`abfss://silver@cacdyamadls.dfs.core.windows.net/carsales`
    WHERE 1=0
    ''')         

df_sink.display()

### Filtering new records and old records

In [0]:
df_filter = df_src.join(df_sink, df_src['Branch_ID'] == df_sink['Branch_ID'], 'left')\
                    .select(df_src['Branch_ID'], df_src['BranchName'], df_sink['dim_branch_key'])

In [0]:
df_filter.display()

****df_filter_old****

In [0]:
df_filter_old = df_filter.filter(col('dim_branch_key').isNotNull())

df_filter_old.display()

****df_filter_new****

In [0]:
df_filter_new = df_filter.filter(col('dim_branch_key').isNull()).select(df_src['Branch_ID'],df_src['BranchName'])

df_filter_new.display()

### Creation of Surrogate Key

***Fetch the max surrogate key from existing table***

In [0]:
if (incremental_flag == '0'):
    max_value = 1 
else:
    max_value_df = spark.sql("SELECT MAX(dim_branch_key) FROM cars_catalog.gold.dim_branch")
    max_value = max_value_df.collect()[0][0]

***Create Surrogate Key Column and ADD the max surrogate key***

In [0]:
df_filter_new = df_filter_new.withColumn('dim_model_key',max_value+monotonically_increasing_id())

In [0]:
df_filter_new.display()

#### Create Findal DF - df_filter_old + df_filter_new

In [0]:
df_final = df_filter_new.union(df_filter_old)

In [0]:
df_final.display()

### Slowly Changing Dimension

#### SCD TYPE - 1 (UPSERT)

In [0]:
from delta.tables import DeltaTable

In [0]:
from delta.tables import DeltaTable

# Configuration
user = "ramos.williamlaurence@gmail.com"
catalog_name = "cars_catalog"
schema_name = "gold"
table_name = "dim_model"
full_table_name = f"{catalog_name}.{schema_name}.{table_name}"
storage_path = "abfss://gold@cacdyamadls.dfs.core.windows.net/dim_model"

if spark.catalog.tableExists(full_table_name):

    # Incremental Run: Update existing records and insert new records
    delta_tbl = DeltaTable.forName(spark, full_table_name)

    (
        delta_tbl.alias("trg")
        .merge(
            df_final.alias("src"),
            "trg.dim_model_key = src.dim_model_key"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:

    # Initial Run: Create the Delta table
    (
        df_final.write
        .format("delta")
        .mode("overwrite")
        .option("path", storage_path)
        .saveAsTable(full_table_name)
    )


In [0]:
%sql
SELECT * FROM cars_catalog.gold.dim_model